# geoarrow_pandas

## Example 1: the GeoArrow pandas dtype and its string form

In [ ]:
# Example 1: GeoArrowExtensionDtype names the geoarrow type and parses back from its string
import geoarrow.pandas as gapd
import geoarrow.pyarrow as ga

dtype = gapd.GeoArrowExtensionDtype(ga.point())
assert str(dtype) == "geoarrow.point" and gapd.GeoArrowExtensionDtype.construct_from_string(str(dtype)) == dtype
z = gapd.GeoArrowExtensionDtype(ga.point().with_dimensions(ga.Dimensions.XYZ))
assert str(z) == "geoarrow.point[z]"
print("Example 1:", dtype, "|", z)

## Example 2: a scalar holds one geometry as WKB, shown as WKT

In [ ]:
# Example 2: GeoArrowExtensionScalar from WKT or from an array element
import geoarrow.pandas as gapd
import geoarrow.pyarrow as ga

scalar = gapd.GeoArrowExtensionScalar("POINT (0 1)")
assert scalar.wkt == "POINT (0 1)" and isinstance(scalar.wkb, bytes)
assert gapd.GeoArrowExtensionScalar(scalar.wkb) == scalar
array = ga.as_geoarrow(["POINT (0 1)", "POINT (1 2)"])
assert gapd.GeoArrowExtensionScalar(array, 1) == gapd.GeoArrowExtensionScalar("POINT (1 2)")
print("Example 2:", repr(scalar), len(scalar.wkb), "WKB bytes")

## Example 3: the .geoarrow accessor converts a Series

In [ ]:
# Example 3: a Series of WKT strings becomes WKB- and point-typed GeoArrow Series
import pandas as pd
import geoarrow.pyarrow as ga
import geoarrow.pandas

s = pd.Series(["POINT (0 1)", "POINT (3 4)"])
assert isinstance(s.geoarrow.as_wkb().dtype.pyarrow_dtype, ga.WkbType)
points = s.geoarrow.as_geoarrow()
assert isinstance(points.dtype.pyarrow_dtype, ga.PointType)
assert points.geoarrow.format_wkt().tolist() == ["POINT (0 1)", "POINT (3 4)"]
print("Example 3:", points.dtype, points.geoarrow.format_wkt().tolist())

## Example 4: bounds and point coordinates

In [ ]:
# Example 4: bounds() gives a DataFrame per row, total_bounds() one row, point_coords() the x/y arrays
import numpy as np
import pandas as pd
import geoarrow.pandas

s = pd.Series(["POINT (0 1)", "POINT (1 2)"])
total = s.geoarrow.total_bounds()
assert (total.xmin[0], total.ymin[0], total.xmax[0], total.ymax[0]) == (0, 1, 1, 2)
assert s.geoarrow.bounds().shape == (2, 4)
x, y = s.geoarrow.point_coords()
assert np.array_equal(np.array(x), [0.0, 1.0]) and np.array_equal(np.array(y), [1.0, 2.0])
print("Example 4:", total.to_dict("records")[0])

## Example 5: set a CRS and an edge type

In [ ]:
# Example 5: with_crs / with_edge_type change the column's type metadata, not its values
import pandas as pd
import geoarrow.pyarrow as ga
import geoarrow.pandas

s = pd.Series(["LINESTRING (0 0, 10 10)"])
with_crs = s.geoarrow.with_crs(ga.OGC_CRS84)
spherical = s.geoarrow.with_edge_type(ga.EdgeType.SPHERICAL)
assert with_crs.dtype.pyarrow_dtype.crs == ga.OGC_CRS84
assert spherical.dtype.pyarrow_dtype.edge_type == ga.EdgeType.SPHERICAL
print("Example 5:", with_crs.dtype.pyarrow_dtype.crs, spherical.dtype.pyarrow_dtype.edge_type)